# Pipeline do *framework* — passo a passo

Percurso completo da cadeia implementada, ancorado no **cenário 1** (linha de produção
serial: Entrada → Processo 1 → Processo 2 → Processo 3 → Saída) e no **formato real de
planilha** trocado com o ProModel.

O ponto de partida não é sintético: é o cockpit
`runs/fake_run_scenario_01_120926/cockpit_nivel_1_v3.xlsx`. Ele **não** é o resultado de
uma campanha de verdade — é um *exemplar* do formato, montado à mão para fixar quais
colunas existem, em que ordem e com que tipo. É dele que sai o contrato que o DoE precisa
cumprir.

**Roteiro**

1. Auditar o cockpit exemplar — o que cada coluna é, e o que nela ainda não é DoE
2. Derivar o **contrato de formato** da parte de ENTRADA da planilha
3. Escolher **3 fatores** para variar e declarar seus limites inteiros
4. Gerar o plano por **Monte Carlo**, **LHS** e **Sobol** (128 pontos) e medir o preenchimento
5. Normalizar e inspecionar a distribuição em **três gráficos 3D interativos**
6. Escolher o método pelas métricas de dispersão e expandir em **replicações** (128 × 2 = 256)
7. Gravar o cockpit **sem respostas** em `runs/test_run_scenario_01_120926/`
8. Seguir o pipeline: simulação fingida, teto de ruído, pré-processamento, metamodelo

> As Fases 4 (avaliação) e 5 (explicabilidade) ainda não estão implementadas; o notebook
> termina no modelo treinado.

In [1]:
import sys, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio

warnings.filterwarnings("ignore")
sys.path.insert(0, str(Path.cwd().parent / "src"))   # notebooks/ -> src/

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 40)

# mimetype: renderiza no VS Code/Jupyter; connected: fallback HTML via CDN, sem
# embutir ~3 MB de plotly.js dentro do .ipynb a cada figura
pio.renderers.default = "plotly_mimetype+notebook_connected"

SEED = 42

RUNS = Path.cwd().parent / "runs"
RUN_EXEMPLO = RUNS / "fake_run_scenario_01_120926"   # cockpit exemplar (formato)
RUN_DESTINO = RUNS / "test_run_scenario_01_120926"   # onde o DoE desta rodada é gravado

print("exemplo :", RUN_EXEMPLO.name, "|", [p.name for p in RUN_EXEMPLO.iterdir()])
print("destino :", RUN_DESTINO.name)

exemplo : fake_run_scenario_01_120926 | ['cockpit_nivel_1_v3.xlsx']
destino : test_run_scenario_01_120926


---
## 1. O cockpit exemplar — auditoria antes de qualquer coisa

A planilha tem **três linhas de cabeçalho**: uma em branco, uma com as faixas de bloco
(`Entrada` / `Saída`) e a terceira com os nomes das colunas. Os dados começam na quarta.
A faixa é a fronteira entre o que **nós** fornecemos e o que o **ProModel** devolve.

In [2]:
CAMINHO_EXEMPLO = RUN_EXEMPLO / "cockpit_nivel_1_v3.xlsx"

bruto = pd.read_excel(CAMINHO_EXEMPLO, sheet_name="cockpit", header=None)
cabecalho = bruto.iloc[2].tolist()

# a faixa da linha 2 marca onde cada bloco COMEÇA
faixas = {int(j): v for j, v in enumerate(bruto.iloc[1]) if isinstance(v, str)}
print("linhas de dados :", len(bruto) - 3)
print("faixas (coluna -> rótulo):", faixas)

inicio_entrada = min(j for j, v in faixas.items() if v == "Entrada")
inicio_saida = min(j for j, v in faixas.items() if v == "Saída")

COL_ID = cabecalho[0]                              # 'Replication ID' — fora das faixas
ENTRADA = cabecalho[inicio_entrada:inicio_saida]   # o que alimentamos
SAIDA = cabecalho[inicio_saida:]                   # o que o ProModel preenche

print(f"\ncoluna de id   ( 1): {COL_ID}")
print(f"bloco ENTRADA  ({len(ENTRADA):2d}): {ENTRADA}")
print(f"bloco SAÍDA    ({len(SAIDA):2d}): {SAIDA}")

linhas de dados : 999
faixas (coluna -> rótulo): {1: 'Entrada', 11: 'Saída'}

coluna de id   ( 1): Replication ID
bloco ENTRADA  (10): ['Item', 'Capacidade M1', 'Quantidade chegada', 'Ocorrências chegada', 'Frequência chegada', 'Tempo processo 1', 'Capacidade M2', 'Capacidade M3', 'Tempo processo 2', 'Tempo processo 3']
bloco SAÍDA    (13): ['Entradas', 'Prontas', 'Tempo_ate_P1', 'Tempo_ate_P2', 'Tempo_ate_P3', 'Tempo_P1', 'Tempo_P2', 'Tempo_P3', 'Feito_P1', 'Feito_P2', 'Feito_P3', 'Tempo_total', 'Tempo_simul']


Repare onde a faixa `Entrada` começa: na **segunda** coluna. `Replication ID` fica de fora
das duas faixas — não é fator do modelo nem KPI, é **bookkeeping**. É por isso que ela
recebe uma inspeção à parte, na Seção 1.2.

In [3]:
cockpit_exemplo = pd.read_excel(CAMINHO_EXEMPLO, sheet_name="cockpit", header=2)

def diagnostico(s: pd.Series) -> str:
    """Classifica o conteúdo OBSERVADO de uma coluna do cockpit exemplar."""
    if s.isna().all():
        return "VAZIA"
    if s.nunique(dropna=True) == 1:
        return "constante"
    if s.dtype.kind in "iu" and np.array_equal(s.to_numpy(), np.arange(1, len(s) + 1)):
        return "índice 1..n (não é fator)"
    return "varia"

auditoria = pd.DataFrame([
    {
        "coluna": c,
        "bloco": "id" if c == COL_ID else ("entrada" if c in ENTRADA else "saída"),
        "dtype": str(cockpit_exemplo[c].dtype),
        "n_únicos": int(cockpit_exemplo[c].nunique(dropna=True)),
        "mín": pd.to_numeric(cockpit_exemplo[c], errors="coerce").min(),
        "máx": pd.to_numeric(cockpit_exemplo[c], errors="coerce").max(),
        "diagnóstico": diagnostico(cockpit_exemplo[c]),
    }
    for c in cabecalho
])
auditoria

,coluna,bloco,dtype,n_únicos,mín,máx,diagnóstico
0,Replication ID,id,float64,0,NaN,NaN,VAZIA
1,Item,entrada,str,999,NaN,NaN,varia
2,Capacidade M1,entrada,int64,1,1.0,1.0,constante
3,Quantidade chegada,entrada,int64,1,1.0,1.0,constante
4,Ocorrências chegada,entrada,int64,999,1.0,999.0,índice 1..n (não é fator)
5,Frequência chegada,entrada,int64,1,1.0,1.0,constante
6,Tempo processo 1,entrada,int64,1,5.0,5.0,constante
7,Capacidade M2,entrada,int64,1,1.0,1.0,constante
8,Capacidade M3,entrada,int64,1,1.0,1.0,constante
9,Tempo processo 2,entrada,int64,1,6.0,6.0,constante


### 1.1 O que a auditoria mostra

**(a) Todo fator de entrada é inteiro.** As nove colunas numéricas do bloco `Entrada` são
`int64` — capacidade é contagem de máquina, tempo de ciclo está em minutos cheios,
quantidade e frequência de chegada são contagens. Nenhum fator contínuo. Isso determina o
tipo de `ParameterSpec` na Seção 3 (**todos `integer`**) e é o que torna relevante a
decodificação por `floor`, e não por arredondamento, feita no módulo de DoE: com
arredondamento, os níveis **extremos** de cada fator receberiam metade da probabilidade
dos internos — justamente as bordas que um DoE precisa cobrir.

**(b) Seis das nove entradas são constantes.** `Capacidade M1/M2/M3 = 1`,
`Quantidade chegada = 1`, `Frequência chegada = 1`, e os tempos de processo fixos em
5, 6 e 7 minutos. Isso não é um DoE: é **uma** configuração repetida 999 vezes.

**(c) A que varia, `Ocorrências chegada`, não varia como fator.** Ela é exatamente
`1, 2, 3, …, 999`: um índice de linha ocupando o lugar de um fator. Nenhum método de
amostragem produziria uma sequência perfeitamente monótona em 999 pontos.

**(d) `Replication ID` está inteiramente vazia.** É a coluna cuja semântica precisa ser
fixada antes de gerar qualquer coisa — é o assunto da Seção 1.2.

Conclusão: o arquivo vale como **especificação de formato**, não como dado. As colunas, a
ordem e os tipos são o contrato; os valores, não.

In [4]:
num_entrada = [c for c in ENTRADA if c != "Item"]
print("Replication ID — células preenchidas:",
      int(cockpit_exemplo[COL_ID].notna().sum()), "de", len(cockpit_exemplo))
print("Ocorrências chegada == 1..999 ?",
      np.array_equal(cockpit_exemplo["Ocorrências chegada"].to_numpy(), np.arange(1, 1000)))
print("combinações distintas das 9 entradas numéricas:",
      cockpit_exemplo[num_entrada].drop_duplicates().shape[0], "em 999 linhas")
print("... ignorando o índice:",
      cockpit_exemplo[[c for c in num_entrada if c != "Ocorrências chegada"]]
      .drop_duplicates().shape[0], "em 999 linhas")

Replication ID — células preenchidas: 0 de 999
Ocorrências chegada == 1..999 ? True
combinações distintas das 9 entradas numéricas: 999 em 999 linhas
... ignorando o índice: 1 em 999 linhas


### 1.2 A semântica de `Replication ID`

`Replication ID` guarda o **id do ponto de origem da replicação**: todas as réplicas de um
mesmo cenário — de um mesmo vetor de entradas — carregam **o mesmo id**. É o que permite,
na volta, agrupar as execuções que compartilham o mesmo $X$ e separar variação *entre*
pontos de variação *entre réplicas do mesmo ponto*. Sem essa coluna não há teto de ruído
nem split agrupado (Seções 9 e 10).

No `sed_meta` essa coluna já existe com outro nome: é o `point_id` de
`replication.build_execution_plan()`. `Replication ID` é o nome que o ProModel enxerga,
`point_id` é o nome interno, e **são a mesma coisa** — o notebook faz a tradução explícita,
nas duas direções, na Seção 7.

Não confundir com o `replication_id` interno (`0..R-1`), que distingue as réplicas *dentro*
de um ponto. Com 999 linhas e 10 réplicas por ponto, a coluna teria ~100 ids distintos,
cada um repetido 10 vezes:

In [5]:
demo = pd.DataFrame({
    "linha do cockpit":            np.arange(1, 26),
    "Replication ID (= point_id)": np.repeat(np.arange(1, 4), 10)[:25],
    "réplica dentro do ponto":     np.tile(np.arange(10), 3)[:25],
})
demo.T

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24
linha do cockpit,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25
Replication ID (= point_id),1,1,1,1,1,1,1,1,1,1,2,2,2,2,2,2,2,2,2,2,3,3,3,3,3
réplica dentro do ponto,0,1,2,3,4,5,6,7,8,9,0,1,2,3,4,5,6,7,8,9,0,1,2,3,4


---
## 2. O contrato que o DoE precisa cumprir

Do cockpit exemplar sai a **parte de ENTRADA** — e só ela. As colunas de saída existem no
contrato, mas nascem vazias: célula em branco é o marcador de "ainda não executado" que
`sim_interface/excel_bridge.py` usa para saber o que o ProModel já rodou.

O contrato abaixo é **lido do arquivo**, não digitado aqui: se o exemplar mudar, ele não
sai de sincronia em silêncio.

In [6]:
CONTEUDO_ESPERADO = {
    "Replication ID":      "id do ponto do DoE; constante entre as réplicas do mesmo ponto",
    "Item":                "rótulo da execução (uma linha = uma execução)",
    "Capacidade M1":       "nº de postos paralelos no Processo 1",
    "Quantidade chegada":  "tamanho do lote que chega por evento de chegada",
    "Ocorrências chegada": "nº de eventos de chegada na rodada (tamanho da campanha)",
    "Frequência chegada":  "intervalo entre chegadas (min)",
    "Tempo processo 1":    "tempo de ciclo do Processo 1 (min)",
    "Capacidade M2":       "nº de postos paralelos no Processo 2",
    "Capacidade M3":       "nº de postos paralelos no Processo 3",
    "Tempo processo 2":    "tempo de ciclo do Processo 2 (min)",
    "Tempo processo 3":    "tempo de ciclo do Processo 3 (min)",
}

contrato = pd.DataFrame([
    {"posição": i + 1, "coluna": c,
     "dtype exigido": "str" if c == "Item" else "int64",
     "conteúdo": CONTEUDO_ESPERADO[c]}
    for i, c in enumerate([COL_ID, *ENTRADA])
])
contrato

,posição,coluna,dtype exigido,conteúdo
0,1,Replication ID,int64,id do ponto do DoE; constante entre as réplica...
1,2,Item,str,rótulo da execução (uma linha = uma execução)
2,3,Capacidade M1,int64,nº de postos paralelos no Processo 1
3,4,Quantidade chegada,int64,tamanho do lote que chega por evento de chegada
4,5,Ocorrências chegada,int64,nº de eventos de chegada na rodada (tamanho da...
5,6,Frequência chegada,int64,intervalo entre chegadas (min)
6,7,Tempo processo 1,int64,tempo de ciclo do Processo 1 (min)
7,8,Capacidade M2,int64,nº de postos paralelos no Processo 2
8,9,Capacidade M3,int64,nº de postos paralelos no Processo 3
9,10,Tempo processo 2,int64,tempo de ciclo do Processo 2 (min)


A **ordem** das colunas é parte do contrato: o ProModel é configurado por fora do Python
para ler posições fixas da planilha, então renomear ou reordenar quebra a ligação em
silêncio. Daí o contrato ser derivado do arquivo e verificado por `assert`, e não redigitado.

---
## 3. Os três fatores e seus limites

O cenário 1 é uma linha serial de três estações, então o que decide o desempenho é a
**distribuição dos tempos de ciclo** entre elas: qual estação é o gargalo e quão
desbalanceada a linha está. São esses os três fatores variados — os outros seis ficam
fixos, **declarados**, não esquecidos.

| Fator | Mínimo | Máximo | Níveis | Valor no exemplar |
|---|---:|---:|---:|---:|
| `Tempo processo 1` | 2 | 20 | 19 | 5 |
| `Tempo processo 2` | 3 | 24 | 22 | 6 |
| `Tempo processo 3` | 4 | 28 | 25 | 7 |

As faixas são **assimétricas de propósito**. Elas cobrem de "estação folgada" a "estação
saturada" em torno do valor do exemplar, e, por terem amplitudes diferentes entre si,
tornam a normalização da Seção 5 necessária em vez de decorativa: sem normalizar, o
`Tempo processo 3` pesaria mais em qualquer métrica de distância só por variar numa faixa
maior.

A única entrada fixa que muda de valor em relação ao exemplar é `Ocorrências chegada`:
lá ela era um índice; aqui assume o papel que o nome promete — o tamanho da campanha,
fixado em 100 chegadas por execução.

In [7]:
from sed_meta.doe.parameter_space import ParameterSpec, generate_design

FATORES = [
    ParameterSpec("Tempo processo 1", "integer", low=2, high=20),
    ParameterSpec("Tempo processo 2", "integer", low=3, high=24),
    ParameterSpec("Tempo processo 3", "integer", low=4, high=28),
]

FIXOS = {
    "Capacidade M1":       1,
    "Quantidade chegada":  1,
    "Ocorrências chegada": 100,   # no exemplar era índice; aqui é o tamanho da campanha
    "Frequência chegada":  1,
    "Capacidade M2":       1,
    "Capacidade M3":       1,
}

N_PONTOS, N_REP = 128, 2          # 128 x 2 = 256 execuções

# o contrato tem de ser coberto por inteiro: nada de fator esquecido fora da lista
assert {s.name for s in FATORES} | set(FIXOS) == set(ENTRADA) - {"Item"}, \
    "fatores variados + fixos precisam cobrir exatamente o bloco de entrada"

pd.DataFrame(
    [{"coluna": s.name, "papel": "FATOR", "domínio": f"[{s.low}, {s.high}]",
      "níveis": int(s.high - s.low + 1)} for s in FATORES]
    + [{"coluna": c, "papel": "fixo", "domínio": "{" + str(v) + "}", "níveis": 1}
       for c, v in FIXOS.items()]
)

,coluna,papel,domínio,níveis
0,Tempo processo 1,FATOR,"[2, 20]",19
1,Tempo processo 2,FATOR,"[3, 24]",22
2,Tempo processo 3,FATOR,"[4, 28]",25
3,Capacidade M1,fixo,{1},1
4,Quantidade chegada,fixo,{1},1
5,Ocorrências chegada,fixo,{100},1
6,Frequência chegada,fixo,{1},1
7,Capacidade M2,fixo,{1},1
8,Capacidade M3,fixo,{1},1


---
## 4. Amostragem: Monte Carlo, LHS e Sobol

Os três métodos previstos na metodologia, sobre o mesmo espaço e a mesma semente. O
**Monte Carlo é o *baseline*** contra o qual os outros dois são comparados — não é um
concorrente, é a referência do que se ganha ao abandonar a amostragem aleatória simples.

São **128 pontos**, potência de 2. A escolha não é estética: as propriedades de
balanceamento de uma sequência de Sobol valem para $n = 2^k$, e com outro $n$ a sequência
perde parte da uniformidade que justifica usá-la. Como o objetivo aqui é comparar os três
em pé de igualdade, os três usam o mesmo $n$.

Isso substitui as 999 linhas do exemplar por 256 execuções (128 pontos × 2 réplicas, na
Seção 6) — menos linhas, cobrindo um espaço que o exemplar não cobria de forma alguma.

In [8]:
designs = {
    metodo: generate_design(FATORES, n_points=N_PONTOS, seed=SEED, method=metodo)
    for metodo in ("monte_carlo", "lhs", "sobol")
}

resumo_designs = pd.DataFrame([
    {"método": m, "linhas": len(d),
     "pontos distintos": len(d.drop_duplicates()),
     "duplicados": int(d.duplicated().sum()),
     "dtypes": ", ".join(sorted(set(d.dtypes.astype(str))))}
    for m, d in designs.items()
]).set_index("método")

# os três precisam ser genuinamente distintos entre si
print("monte_carlo == lhs ?  ",
      np.array_equal(designs["monte_carlo"].to_numpy(), designs["lhs"].to_numpy()))
print("lhs == sobol ?        ",
      np.array_equal(designs["lhs"].to_numpy(), designs["sobol"].to_numpy()))
resumo_designs

monte_carlo == lhs ?   False
lhs == sobol ?         False


,linhas,pontos distintos,duplicados,dtypes
método,,,,
monte_carlo,128,127,1,int64
lhs,128,128,0,int64
sobol,128,128,0,int64


Primeiro sintoma da diferença entre os métodos, antes de qualquer métrica: em espaço
**inteiro** o Monte Carlo repete ponto. Duas linhas idênticas do design não são duas
observações independentes do espaço — são réplicas não solicitadas de um mesmo cenário,
gastando orçamento de simulação sem cobrir nada de novo. LHS e Sobol não colidem.

In [9]:
designs["lhs"].head()

,Tempo processo 1,Tempo processo 2,Tempo processo 3
0,18,7,17
1,12,16,13
2,12,10,18
3,5,22,25
4,15,12,14


### 4.1 Métricas de preenchimento e dispersão

Todas calculadas sobre o espaço **normalizado para $[0,1]^3$** pelos limites *declarados*
no `ParameterSpec` — não pelos observados. Fosse pelos observados, um design que não
alcançasse as bordas pareceria artificialmente bem espalhado dentro da própria casca.

| Métrica | O que mede | Direção |
|---|---|---|
| Distância mínima entre pontos | O par mais próximo — detecta aglomeração | **maior é melhor** |
| Distância média ao vizinho mais próximo | Espaçamento típico | **maior é melhor** |
| Desvio da distância ao vizinho | Regularidade do espaçamento | menor é melhor |
| Distância média entre todos os pares | Espalhamento global | maior é melhor |
| Cobertura dos vértices | Distância média de cada vértice do cubo ao ponto mais próximo — detecta vazios nos **cantos** | **menor é melhor** |
| Discrepância $L^2$ estrela | Desvio da distribuição uniforme ideal (medida clássica de QMC) | menor é melhor |
| Dispersão dos espaçamentos marginais | Uniformidade **por fator** — é o que o LHS garante por construção | menor é melhor |

In [10]:
from itertools import product
from scipy.spatial.distance import pdist, squareform
from scipy.stats import qmc


def para_cubo_unitario(df, specs):
    """Normaliza pelos limites DECLARADOS no spec, não pelos observados."""
    return np.column_stack([
        (df[s.name].to_numpy(float) - s.low) / (s.high - s.low) for s in specs
    ])


def metricas_preenchimento(df, specs):
    U = para_cubo_unitario(df, specs)
    n, d = U.shape

    D = squareform(pdist(U))
    np.fill_diagonal(D, np.inf)
    vizinho = D.min(axis=1)

    # distância de cada vértice do hipercubo ao ponto mais próximo do design
    vertices = np.array(list(product([0.0, 1.0], repeat=d)))
    dist_vertices = np.min(
        np.linalg.norm(vertices[:, None, :] - U[None, :, :], axis=2), axis=1
    )

    # uniformidade marginal: dispersão dos espaçamentos, por fator
    disp_marginal = float(np.mean([
        np.std(np.diff(np.sort(U[:, j]))) for j in range(d)
    ]))

    return {
        "dist. mín. entre pontos":  D.min(),
        "dist. média ao vizinho":   vizinho.mean(),
        "desvio dist. ao vizinho":  vizinho.std(),
        "dist. média entre pares":  pdist(U).mean(),
        "cobertura dos vértices":   dist_vertices.mean(),
        "discrepância L2-estrela":  qmc.discrepancy(U, method="L2-star"),
        "disp. espaç. marginal":    disp_marginal,
    }


tabela = pd.DataFrame({m: metricas_preenchimento(d, FATORES) for m, d in designs.items()})
tabela.round(4)

,monte_carlo,lhs,sobol
dist. mín. entre pontos,0.0000,0.1179,0.0633
dist. média ao vizinho,0.1223,0.1750,0.1591
desvio dist. ao vizinho,0.0451,0.0232,0.0416
dist. média entre pares,0.6862,0.6966,0.6985
cobertura dos vértices,0.1701,0.1647,0.2061
discrepância L2-estrela,0.0211,0.0178,0.0141
disp. espaç. marginal,0.0178,0.0178,0.0178


**Como ler.** Nas quatro primeiras linhas, LHS e Sobol devem bater o Monte Carlo: a
amostragem aleatória aglomera pontos e deixa vazios, o que aparece como distância mínima
pequena (aqui, **zero** — o par duplicado) e espaçamento irregular. A linha dos vértices
mostra quem cobre melhor os **cantos**, região crítica num DoE e a que a amostragem
aleatória mais frequentemente deixa descoberta.

A última linha merece atenção: **a dispersão marginal é idêntica nos três métodos**. Não é
erro de cálculo, é consequência de o espaço ser inteiro. Com 128 pontos sobre fatores de
19, 22 e 25 níveis, os três designs ocupam **todos** os níveis de **todos** os fatores, e
o conjunto dos espaçamentos marginais passa a ser determinado pela grade, não pelo método.
A métrica que o LHS normalmente vence por construção fica saturada — e a discriminação
migra inteira para as métricas **conjuntas**, que enxergam a estrutura tridimensional.

In [11]:
cobertura_marginal = pd.DataFrame({
    m: {s.name: f"{d[s.name].nunique()} / {int(s.high - s.low + 1)}" for s in FATORES}
    for m, d in designs.items()
})
print("níveis ocupados / níveis existentes, por fator:")
print(cobertura_marginal.to_string(), "\n")
print("dispersão marginal (10 casas):")
print(tabela.loc["disp. espaç. marginal"].apply(lambda v: f"{v:.10f}").to_string())

níveis ocupados / níveis existentes, por fator:
                 monte_carlo      lhs    sobol
Tempo processo 1     19 / 19  19 / 19  19 / 19
Tempo processo 2     22 / 22  22 / 22  22 / 22
Tempo processo 3     25 / 25  25 / 25  25 / 25 

dispersão marginal (10 casas):
monte_carlo    0.0177929878
lhs            0.0177929878
sobol          0.0177929878


---
## 5. Normalização e inspeção visual do preenchimento

A verificação visual da distribuição. Os três fatores vão para $[0,1]$ pelos limites
declarados, de modo que os eixos fiquem comparáveis entre si — sem isso o
`Tempo processo 3`, que varia numa faixa maior, dominaria a leitura do gráfico do mesmo
jeito que dominaria as distâncias.

### Por que 16 pontos, e não os 128

Os 128 pontos do plano real são **densos demais para comparar métodos a olho**: num cubo
com 128 marcadores, os três desenhos parecem igualmente cheios, e a diferença entre eles —
que está nos vazios e nos grumos — desaparece na sobreposição. Com **16 pontos** cada
marcador é individualmente visível, e dá para contar os vazios.

16 é potência de 2, então a comparação continua justa com o Sobol (mesma ressalva da
Seção 4). O espaço, a semente e os limites são os mesmos: **muda só o $n$**. Estes 16
pontos existem para ver; quem vai para a planilha continua sendo o design de 128 da
Seção 4 — mostrado no fim desta seção.

Cada figura é **rotacionável** (arraste com o botão esquerdo; a roda dá zoom). Para
combater a ambiguidade de profundidade que todo gráfico 3D tem, cada ponto recebe uma
**haste até o piso** e uma **sombra projetada** nele: a sombra é a posição do ponto no
plano (`Tempo processo 1`, `Tempo processo 2`), e é nela que aglomeração e vazio aparecem
mesmo sem girar a figura.

In [12]:
N_VISUAL = 16          # potência de 2, pelo mesmo motivo dos 128

designs_visual = {
    metodo: generate_design(FATORES, n_points=N_VISUAL, seed=SEED, method=metodo)
    for metodo in ("monte_carlo", "lhs", "sobol")
}

CORES = {"monte_carlo": "#D1495B", "lhs": "#1F77B4", "sobol": "#2A9D8F"}
EIXOS = [s.name for s in FATORES]

normalizados = {m: para_cubo_unitario(d, FATORES) for m, d in designs_visual.items()}
tabela_visual = pd.DataFrame({
    m: metricas_preenchimento(d, FATORES) for m, d in designs_visual.items()
})

print(f"as mesmas métricas da Seção 4.1, agora com n = {N_VISUAL}:")
tabela_visual.round(4)

as mesmas métricas da Seção 4.1, agora com n = 16:


,monte_carlo,lhs,sobol
dist. mín. entre pontos,0.1187,0.3829,0.2557
dist. média ao vizinho,0.2624,0.4309,0.3551
desvio dist. ao vizinho,0.1266,0.0232,0.0589
dist. média entre pares,0.6869,0.7277,0.7198
cobertura dos vértices,0.4112,0.3877,0.3815
discrepância L2-estrela,0.0640,0.0395,0.0441
disp. espaç. marginal,0.0577,0.0367,0.0375


Com $n = 16$ a métrica marginal **volta a discriminar**: 16 pontos não chegam a ocupar
todos os 19, 22 e 25 níveis de cada fator, então a grade inteira deixa de ditar os
espaçamentos e o que o LHS garante por construção reaparece. É o contraponto exato da
saturação observada com $n = 128$ na Seção 4.1.

In [13]:
# --- estilo comum das figuras 3D -------------------------------------
PANO, GRADE, TEXTO, SOMBRA = "#F2F3F5", "#C9CDD4", "#2B2F36", "#9AA0A6"
PISO = -0.06


def _eixo(titulo):
    return dict(
        title=dict(text=titulo, font=dict(size=12, color=TEXTO)),
        range=[PISO, 1.06],
        tickmode="array", tickvals=[0.0, 0.25, 0.5, 0.75, 1.0],
        ticktext=["0", "0,25", "0,5", "0,75", "1"],
        tickfont=dict(size=10, color=TEXTO),
        showbackground=True, backgroundcolor=PANO,
        showgrid=True, gridcolor=GRADE, gridwidth=1,
        zeroline=False, showline=True, linecolor=GRADE, linewidth=2,
        ticks="outside", tickcolor=GRADE, ticklen=4,
    )


def _hastes(U):
    # segmentos verticais do piso ate cada ponto — e o que da profundidade
    topo = np.maximum(U[:, 2] - 0.018, PISO)      # para um fio abaixo do marcador
    x = np.repeat(U[:, 0], 3)
    y = np.repeat(U[:, 1], 3)
    z = np.column_stack([np.full(len(U), PISO), topo, np.full(len(U), np.nan)]).ravel()
    x[2::3] = np.nan                              # NaN separa um segmento do próximo
    y[2::3] = np.nan
    return x, y, z


def _layout(titulo, subtitulo, largura=600, altura=530, **extra):
    return dict(
        template="plotly_white",
        title=dict(
            text=f"<b>{titulo}</b><br><span style='font-size:11px;color:#6B7280'>{subtitulo}</span>",
            x=0.5, xanchor="center", y=0.97, yanchor="top",
            font=dict(size=15, color=TEXTO),
        ),
        width=largura, height=altura, margin=dict(l=10, r=10, t=62, b=20),
        paper_bgcolor="white", font=dict(color=TEXTO, size=11),
        scene=dict(
            xaxis=_eixo(f"{EIXOS[0]} (norm.)"),
            yaxis=_eixo(f"{EIXOS[1]} (norm.)"),
            zaxis=_eixo(f"{EIXOS[2]} (norm.)"),
            camera=dict(eye=dict(x=1.58, y=1.5, z=0.95), center=dict(x=0, y=0, z=-0.08)),
            aspectmode="cube",
        ),
        **extra,
    )


HOVER = (f"<b>{EIXOS[0]}</b>: %{{customdata[0]}} min<br>"
         f"<b>{EIXOS[1]}</b>: %{{customdata[1]}} min<br>"
         f"<b>{EIXOS[2]}</b>: %{{customdata[2]}} min<extra></extra>")


def pontos_3d(U, valores, cor, nome, tamanho=6.5, opacidade=0.95, sombra=0.28):
    return go.Scatter3d(
        x=U[:, 0], y=U[:, 1], z=U[:, 2], mode="markers", name=nome,
        marker=dict(size=tamanho, color=cor, opacity=opacidade,
                    line=dict(width=1.0, color="white")),
        projection=dict(z=dict(show=True, opacity=sombra, scale=0.6)),
        customdata=valores, hovertemplate=HOVER,
    )


def hastes_3d(U):
    hx, hy, hz = _hastes(U)
    return go.Scatter3d(x=hx, y=hy, z=hz, mode="lines", hoverinfo="skip",
                        showlegend=False, line=dict(color=SOMBRA, width=1))


def figura_design(U, valores, metodo, titulo, tamanho=6.5):
    metricas = tabela_visual if len(U) == N_VISUAL else tabela
    vira = lambda v: f"{v}".replace(".", ",")          # decimal com vírgula, como nos eixos
    sub = (f"distância mínima {vira(round(metricas.loc['dist. mín. entre pontos', metodo], 3))}"
           f" · discrepância L2* {vira(round(metricas.loc['discrepância L2-estrela', metodo], 4))}")
    fig = go.Figure([hastes_3d(U), pontos_3d(U, valores, CORES[metodo], metodo, tamanho)])
    fig.update_layout(**_layout(titulo, sub), showlegend=False)
    return fig


figuras = {
    metodo: figura_design(U, designs_visual[metodo][EIXOS].to_numpy(), metodo,
                          f"{metodo} — {N_VISUAL} pontos")
    for metodo, U in normalizados.items()
}

for metodo in ("monte_carlo", "lhs", "sobol"):
    figuras[metodo].show()

**O que se vê**, girando as três figuras e comparando as sombras no piso:

- **Monte Carlo** — pares quase colados e vazios grandes. É a assinatura da amostragem
  aleatória simples: nada impede dois sorteios de caírem perto, e cada aglomeração deixa
  necessariamente um buraco em outro lugar. Com 16 pontos isso fica óbvio; com 128 ficava
  escondido pela densidade.
- **LHS** — sem grumos e sem buracos grandes. A estratificação por fator obriga cada faixa
  marginal a receber um ponto, e o critério `ese` empurra o arranjo conjunto para longe de
  si mesmo.
- **Sobol** — o mais regular dos três no interior, com a estrutura em blocos típica de uma
  sequência de baixa discrepância: os pontos preenchem o cubo por subdivisões sucessivas.
  O preço aparece nos cantos, que ele tende a alcançar por último — é a linha "cobertura
  dos vértices" da tabela.

A sobreposição põe os três no mesmo cubo (clique na legenda para isolar um método):

In [14]:
sobreposto = go.Figure([
    pontos_3d(U, designs_visual[metodo][EIXOS].to_numpy(), CORES[metodo], metodo,
              tamanho=6.0, opacidade=0.9, sombra=0.22)
    for metodo, U in normalizados.items()
])
sobreposto.update_layout(**_layout(
    f"os três métodos sobrepostos — {N_VISUAL} pontos cada",
    "mesma semente, mesmo espaço; muda só o método de amostragem",
    largura=680, altura=620,
    legend=dict(orientation="h", y=0.02, x=0.5, xanchor="center",
                bgcolor="rgba(255,255,255,0.7)"),
))
sobreposto.show()

### 5.1 Escolha do método pelas métricas de dispersão

A escolha é decidida sobre o **plano real, o de 128 pontos** — as 16 amostras da seção
anterior servem para enxergar, não para escolher. Um design de 16 pontos num espaço de 3
fatores é pequeno demais para que a diferença entre métodos seja estável: troque a semente
e o pódio pode mudar.

O critério é o **posto médio** nas sete métricas, com o sentido de cada uma respeitado
(três são "maior é melhor", quatro são "menor é melhor"). Usar o posto, e não o valor, é o
que permite somar métricas em unidades incomparáveis — distância, discrepância e desvio —
sem que a de maior escala decida sozinha.

O design escolhido aqui é o que vai para a planilha na Seção 7.

In [15]:
MAIOR_MELHOR = ["dist. mín. entre pontos", "dist. média ao vizinho",
                "dist. média entre pares"]

postos = pd.DataFrame({
    metrica: (tabela.loc[metrica].rank(ascending=(metrica not in MAIOR_MELHOR)))
    for metrica in tabela.index
}).T
postos.loc["POSTO MÉDIO"] = postos.mean()

placar = postos.loc["POSTO MÉDIO"].sort_values()
ESCOLHIDO = str(placar.idxmin())

print(placar.round(3).to_string())
print(f"\nmétodo escolhido: {ESCOLHIDO}")
postos.round(2)

lhs            1.429
sobol          1.857
monte_carlo    2.714

método escolhido: lhs


,monte_carlo,lhs,sobol
dist. mín. entre pontos,3.00,1.00,2.00
dist. média ao vizinho,3.00,1.00,2.00
desvio dist. ao vizinho,3.00,1.00,2.00
dist. média entre pares,3.00,2.00,1.00
cobertura dos vértices,2.00,1.00,3.00
discrepância L2-estrela,3.00,2.00,1.00
disp. espaç. marginal,2.00,2.00,2.00
POSTO MÉDIO,2.71,1.43,1.86


E este é o design que **de fato** vai para o cockpit: o método vencedor, com os 128 pontos
do plano. Vale a comparação direta com a versão de 16 — é a mesma regra de amostragem,
e é justamente a densidade que torna a inspeção visual inconclusiva e as métricas
necessárias.

In [16]:
U_planilha = para_cubo_unitario(designs[ESCOLHIDO], FATORES)

figura_design(U_planilha, designs[ESCOLHIDO][EIXOS].to_numpy(), ESCOLHIDO,
              f"{ESCOLHIDO} — {N_PONTOS} pontos (o design que vai para a planilha)",
              tamanho=4.0).show()

---
## 6. Replicações

O simulador é estocástico, então cada ponto do design é executado mais de uma vez.
`build_execution_plan()` expande os 128 pontos em **128 × 2 = 256 execuções**, uma por
linha, com semente derivada de forma determinística do ponto e do índice da réplica.

Duas réplicas por ponto é o mínimo que ainda permite estimar variância intra-ponto — e
mantém o total em potência de 2, pelo mesmo motivo da Seção 4.

É aqui que nasce o `point_id` que a planilha vai carregar na coluna `Replication ID`.

In [17]:
from sed_meta.doe.replication import (
    ReplicationPolicy, build_execution_plan,
    aggregate_replications, compute_noise_ceiling,
)

design = designs[ESCOLHIDO]
plano = build_execution_plan(design, ReplicationPolicy(n_replications=N_REP, id_base=0))

print(f"{N_PONTOS} pontos x {N_REP} réplicas = {len(plano)} execuções")
print(f"point_id distintos : {plano['point_id'].nunique()}")
print(f"réplicas por ponto : {sorted(map(int, plano.groupby('point_id').size().unique()))}")
print(f"sementes únicas    : {plano['unique_id'].nunique()}")
plano.head(6)

128 pontos x 2 réplicas = 256 execuções
point_id distintos : 128
réplicas por ponto : [2]
sementes únicas    : 256


,point_id,replication_id,seed,Tempo processo 1,Tempo processo 2,Tempo processo 3
0,0,0,0,18,7,17
1,0,1,1,18,7,17
2,1,0,2,12,16,13
3,1,1,3,12,16,13
4,2,0,4,12,10,18
5,2,1,5,12,10,18


As duas primeiras linhas repetem os parâmetros do ponto 0 e mudam só a semente — é
exatamente isso que caracteriza a replicação, e é exatamente isso que `Replication ID`
constante vai registrar na planilha. Os `dtypes` inteiros sobrevivem à expansão:

In [18]:
pd.DataFrame({"dtype": plano.dtypes.astype(str)})

,dtype
point_id,int64
replication_id,int64
seed,int64
Tempo processo 1,int64
Tempo processo 2,int64
Tempo processo 3,int64


---
## 7. O cockpit gravado em `test_run_scenario_01_120926`

Agora o plano vira planilha, no formato auditado na Seção 2: bloco de entrada preenchido,
bloco de saída **vazio**. A tradução entre os dois vocabulários é feita aqui, explicitamente:

| Interno (`sed_meta`) | Planilha (ProModel) |
|---|---|
| `point_id` | `Replication ID` |
| índice da linha | `Item` |
| coluna do `ParameterSpec` | coluna homônima do bloco `Entrada` |
| `replication_id`, `seed` | *não vão para a planilha* |

`replication_id` e `seed` ficam de fora porque **o formato do ProModel não tem essas
colunas** — o exemplar não as tem, e inventá-las quebraria o contrato posicional. Elas são
recuperáveis: `replication_id` é a ordem da réplica dentro do `Replication ID`, e o plano
completo é gravado ao lado, em `plano_execucao.csv`, para não perder a rastreabilidade da
semente.

Duas diferenças deliberadas em relação ao exemplar, ambas documentadas em
`excel_bridge.py`: a planilha gravada tem **uma única linha de cabeçalho** (a faixa
`Entrada`/`Saída` mesclada deslocaria os dados e tornaria ambígua a leitura pelo ProModel —
a distinção fica por cor de preenchimento), e leva uma **aba oculta de esquema**, sem a
qual o round-trip pelo Excel perderia os dtypes.

In [19]:
from sed_meta.sim_interface.excel_bridge import (
    build_cockpit_table, write_cockpit_excel, read_cockpit_excel,
    extract_results, select_complete_runs,
)

ORDEM_ENTRADA = [COL_ID, *ENTRADA]        # a ordem do contrato, lida do exemplar


def montar_entradas(plano: pd.DataFrame) -> pd.DataFrame:
    """Traduz o plano de execução interno para o vocabulário da planilha."""
    entradas = pd.DataFrame(index=range(len(plano)))
    entradas[COL_ID] = plano["point_id"].to_numpy()          # id do ponto de origem
    entradas["Item"] = [f"Teste {i}" for i in range(1, len(plano) + 1)]
    for coluna, valor in FIXOS.items():
        entradas[coluna] = np.int64(valor)
    for spec in FATORES:
        entradas[spec.name] = plano[spec.name].to_numpy()
    return entradas[ORDEM_ENTRADA]                            # ordem = contrato


entradas = montar_entradas(plano)
cockpit = build_cockpit_table(entradas, SAIDA)   # + colunas de KPI vazias

print("formato:", cockpit.shape, "| exemplar:", (len(cockpit_exemplo), len(cabecalho)))
print("colunas iguais às do exemplar, na mesma ordem:",
      list(cockpit.columns) == list(cabecalho))
cockpit.head(6)

formato: (256, 24) | exemplar: (999, 24)
colunas iguais às do exemplar, na mesma ordem: True


,Replication ID,Item,Capacidade M1,Quantidade chegada,Ocorrências chegada,Frequência chegada,Tempo processo 1,Capacidade M2,Capacidade M3,Tempo processo 2,Tempo processo 3,Entradas,Prontas,Tempo_ate_P1,Tempo_ate_P2,Tempo_ate_P3,Tempo_P1,Tempo_P2,Tempo_P3,Feito_P1,Feito_P2,Feito_P3,Tempo_total,Tempo_simul
0,0,Teste 1,1,1,100,1,18,1,1,7,17,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0,Teste 2,1,1,100,1,18,1,1,7,17,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1,Teste 3,1,1,100,1,12,1,1,16,13,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1,Teste 4,1,1,100,1,12,1,1,16,13,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2,Teste 5,1,1,100,1,12,1,1,10,18,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,2,Teste 6,1,1,100,1,12,1,1,10,18,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Conferindo a propriedade que dá sentido à coluna: **id constante dentro do ponto, um id
por ponto**, e nenhuma célula de saída preenchida.

In [20]:
por_id = cockpit.groupby(COL_ID)[[s.name for s in FATORES]].nunique()
print("ids distintos                       :", cockpit[COL_ID].nunique())
print("linhas por id                       :", sorted(map(int, cockpit.groupby(COL_ID).size().unique())))
print("fatores variando DENTRO de um id    :", int((por_id > 1).to_numpy().sum()), "(tem de ser 0)")
print("células de saída preenchidas        :", int(cockpit[SAIDA].notna().to_numpy().sum()), "(tem de ser 0)")
print("dtypes do bloco de entrada          :", sorted(set(cockpit[ORDEM_ENTRADA].dtypes.astype(str))))

ids distintos                       : 128
linhas por id                       : [2]
fatores variando DENTRO de um id    : 0 (tem de ser 0)
células de saída preenchidas        : 0 (tem de ser 0)
dtypes do bloco de entrada          : ['int64', 'str']


In [21]:
RUN_DESTINO.mkdir(parents=True, exist_ok=True)
ARQUIVO = RUN_DESTINO / f"cockpit_scenario_01_{ESCOLHIDO}_{N_PONTOS}x{N_REP}.xlsx"

write_cockpit_excel(cockpit, str(ARQUIVO), entrada_cols=ORDEM_ENTRADA, saida_cols=SAIDA)
plano.to_csv(RUN_DESTINO / "plano_execucao.csv", index=False, encoding="utf-8")

for p in sorted(RUN_DESTINO.iterdir()):
    print(f"{p.name:46s} {p.stat().st_size / 1024:7.1f} KB")

cockpit_scenario_01_lhs_128x2.xlsx                23.1 KB
plano_execucao.csv                                 4.5 KB


### 7.1 Round-trip: a planilha volta como o framework espera?

Ler de volta o arquivo recém-gravado testa as duas pontas de uma vez — que os dtypes
sobreviveram ao Excel, e que o framework reconhece o estado do cockpit. Com o bloco de
saída vazio, **todas** as 256 execuções têm de voltar marcadas como `pendente`: é esse o
estado de um cockpit entregue e ainda não rodado.

A tradução inversa (`Replication ID` → `point_id`) acontece aqui; `replication_id` é
reconstruído pela ordem da réplica dentro do ponto, que é determinística.

In [22]:
volta = read_cockpit_excel(str(ARQUIVO))

interno = volta.rename(columns={COL_ID: "point_id"})
interno["replication_id"] = interno.groupby("point_id").cumcount()

raw_pendente = extract_results(interno, SAIDA)
completas = select_complete_runs(raw_pendente)

print("status das execuções :", raw_pendente["status_execucao"].value_counts().to_dict())
print("execuções completas  :", len(completas), "(nada a agregar ainda — correto)")
print("dtypes preservados   :",
      (volta[ORDEM_ENTRADA].dtypes.astype(str) == cockpit[ORDEM_ENTRADA].dtypes.astype(str)).all())
print("valores idênticos    :",
      volta[ORDEM_ENTRADA].equals(cockpit[ORDEM_ENTRADA]))

select_complete_runs: seguindo com 0 de 256 execuções; 256 fora deste recorte (pendente=256). Nada foi apagado — as linhas continuam no DataFrame de origem e na planilha.


status das execuções : {'pendente': 256}
execuções completas  : 0 (nada a agregar ainda — correto)
dtypes preservados   : True
valores idênticos    : True


O arquivo está pronto para o ProModel: ele varre a planilha linha a linha, lê as onze
colunas de entrada, roda a simulação com aquela configuração e escreve os treze KPIs na
**mesma linha** antes de avançar. A partir daqui, o notebook segue **sem** esse arquivo —
o que vem a seguir é uma simulação fingida em memória, e o cockpit gravado permanece
sem respostas, como deve.

---
## 8. Fingindo que a simulação rodou

Substitui-se aqui a campanha no ProModel por uma resposta sintética, coerente com a linha
serial do cenário 1. Para uma campanha de $n$ peças numa linha de três estações sem
bloqueio, o tempo até a última peça sair é

$$T_{\text{total}} \;=\; \underbrace{(t_1 + t_2 + t_3)}_{\text{enchimento da linha}}
\;+\; (n - 1)\,\underbrace{\max(t_1, t_2, t_3)}_{\text{gargalo}}$$

Duas propriedades dessa forma importam para o metamodelo: ela é **não linear e não suave**
(o `max` troca de estação quando o gargalo muda), e depende dos três fatores de maneira
desigual — o gargalo domina, os outros dois quase só aparecem no enchimento. É um alvo
honesto: fácil o bastante para ser aprendido, difícil o bastante para não ser trivial.

Sobre ela vai ruído **multiplicativo lognormal**, com $\sigma$ crescendo com o gargalo.
Isso reproduz duas características do SED estocástico real:

- assimetria à direita — motiva a transformação logarítmica do alvo na Seção 10;
- **heterocedasticidade** — a dispersão entre réplicas cresce com a carga da estação
  gargalo, e é isso que dá sentido ao peso amostral calculado na Seção 9.

Na campanha real, esta célula é substituída pelo preenchimento da planilha pelo ProModel;
**o resto do notebook não muda**.

In [23]:
rng = np.random.default_rng(SEED)

t1, t2, t3 = (plano[s.name].to_numpy(float) for s in FATORES)
gargalo = np.maximum.reduce([t1, t2, t3])
n_pecas = FIXOS["Ocorrências chegada"]

# dispersão entre réplicas cresce com a carga do gargalo (heterocedasticidade)
sigma = 0.03 + 0.10 * (gargalo / max(s.high for s in FATORES))

resultados = plano.copy()
resultados["Tempo_total"] = (
    60.0 * ((t1 + t2 + t3) + (n_pecas - 1) * gargalo) * rng.lognormal(0.0, sigma)
)
resultados["Tempo_ate_P3"] = 60.0 * (t1 + t2) * rng.lognormal(0.0, 0.5 * sigma)

KPIS = ["Tempo_total", "Tempo_ate_P3"]   # dois dos treze KPIs do contrato
resultados[["point_id", "replication_id", *[s.name for s in FATORES], *KPIS]].head(8)

,point_id,replication_id,Tempo processo 1,Tempo processo 2,Tempo processo 3,Tempo_total,Tempo_ate_P3
0,0,0,18,7,17,112629.866121,1531.061404
1,0,1,18,7,17,99218.134949,1473.634851
2,1,0,12,16,13,104089.264068,1670.232663
3,1,1,12,16,13,105828.078315,1582.312871
4,2,0,12,10,18,90951.337860,1305.261685
5,2,1,12,10,18,96689.270151,1303.526741
6,3,0,5,22,25,153949.853757,1642.588621
7,3,1,5,22,25,146006.941794,1567.222348


A heterocedasticidade da resposta sintética, que é o ponto de simular em vez de resolver
em forma fechada:

In [24]:
por_ponto = resultados.groupby("point_id")["Tempo_total"].agg(["mean", "std"])

disp = go.Figure(go.Scatter(
    x=por_ponto["mean"] / 3600, y=por_ponto["std"] / 3600, mode="markers",
    marker=dict(size=7, color=CORES[ESCOLHIDO], opacity=0.75,
                line=dict(width=0.5, color="white")),
    hovertemplate="média: %{x:.2f} h<br>desvio: %{y:.2f} h<extra></extra>",
))
disp.update_layout(
    title=dict(text="Dispersão entre réplicas cresce com o nível da resposta", x=0.5),
    xaxis_title="média do Tempo_total no ponto (h)",
    yaxis_title="desvio entre as 2 réplicas (h)",
    width=640, height=420, margin=dict(l=60, r=20, t=50, b=50),
    plot_bgcolor="rgba(245,245,247,0.6)",
)
disp.show()

---
## 9. Duas granularidades, e qual delas treina o metamodelo

Neste ponto há uma decisão que muda o que o metamodelo aprende.

**(A) Por replicação** — as 256 linhas que acabamos de produzir. Cada ponto do DoE aparece
$K$ vezes, com **as mesmas entradas e $K$ resultados diferentes**, porque o simulador é
estocástico.

**(B) Por ponto** — consolidando as $K$ réplicas em uma linha, com média e estatísticas de
dispersão.

**O treino usa (A).** As $K$ réplicas são a evidência empírica de que aquela combinação de
parâmetros não produz *um* valor, e sim uma distribuição. Entregá-las ao metamodelo é o que
o força a generalizar a tendência em vez de decorar observações: com $K$ alvos conflitantes
sobre o mesmo $X$, não existe função que passe por todos, e o ajuste é empurrado para a
resposta média **por construção** — sem que nenhuma regularização precise ser imposta de
fora. É também a granularidade que preserva a estrutura de replicação de que métodos com
modelagem explícita de ruído dependem.

**(B) continua necessário**, mas para outra coisa: é dele que sai o teto de ruído.

In [25]:
param_cols = [s.name for s in FATORES]
dataset_agregado = aggregate_replications(resultados, param_cols=param_cols, kpi_cols=KPIS)

print(f"(A) por replicação : {len(resultados):3d} linhas "
      f"({resultados['point_id'].nunique()} pontos x {N_REP} réplicas)  <- treino")
print(f"(B) por ponto      : {len(dataset_agregado):3d} linhas"
      f"                       <- teto de ruído")
dataset_agregado[["point_id", "Tempo_total_mean", "Tempo_total_std",
                  "Tempo_total_n", "Tempo_total_sem"]].head(5).round(1)

(A) por replicação : 256 linhas (128 pontos x 2 réplicas)  <- treino
(B) por ponto      : 128 linhas                       <- teto de ruído


,point_id,Tempo_total_mean,Tempo_total_std,Tempo_total_n,Tempo_total_sem
0,0,105924.0,9483.5,2,6705.9
1,1,104958.7,1229.5,2,869.4
2,2,93820.3,4057.3,2,2869.0
3,3,149978.4,5616.5,2,3971.5
4,4,88345.7,4364.0,2,3085.8


### 9.1 Teto de ruído

Decompõe a variância total em variação **entre pontos** (sinal, que um metamodelo pode
aprender) e **entre réplicas do mesmo ponto** (ruído do simulador, que ninguém pode
prever). A razão é o limite superior do $R^2$ legítimo.

Ele precisa ser calculado **na mesma escala em que o modelo treina** — aqui, a do
logaritmo, já que o alvo usa `log_standard`. Comparar um $R^2$ obtido em escala log contra
um teto calculado na escala bruta seria comparar coisas diferentes.

In [26]:
teto_bruto = compute_noise_ceiling(resultados, KPIS)

em_log = resultados.copy()
em_log["Tempo_total"] = np.log1p(em_log["Tempo_total"])
teto_log = compute_noise_ceiling(em_log, ["Tempo_total"])

print(f"Tempo_total   — teto na escala bruta : {teto_bruto['Tempo_total']:.4f}")
print(f"Tempo_total   — teto na escala log   : {teto_log['Tempo_total']:.4f}  <- a referência")
print(f"Tempo_ate_P3  — teto na escala bruta : {teto_bruto['Tempo_ate_P3']:.4f}")

Tempo_total   — teto na escala bruta : 0.8744
Tempo_total   — teto na escala log   : 0.9205  <- a referência
Tempo_ate_P3  — teto na escala bruta : 0.9773


---
## 10. Pré-processamento

A ordem é deliberada e importa: **partição primeiro, ajuste depois**.

- **Entradas (X):** codificadas a partir dos limites *declarados* no `ParameterSpec` — não
  dos observados. Ajustar em qualquer subconjunto das linhas dá o mesmo resultado.
- **Alvo (y):** a transformação é aprendida, e **só** da partição de treino.

### O split tem de ser agrupado por ponto

Com dados por replicação, uma partição comum seria um erro grave: as $K$ réplicas de um
ponto têm **exatamente** os mesmos valores de entrada, então algumas cairiam no treino e
outras no teste — e o modelo seria avaliado prevendo uma linha de $X$ que já viu. Isso não
é generalização, é consulta.

`split()` detecta que há `point_id` repetido e agrupa sozinho. A detecção é automática
justamente porque o erro seria silencioso.

In [27]:
from sed_meta.preprocessing import PreprocessingPipeline, TargetTransformSpec

alvos = [
    TargetTransformSpec("Tempo_total",  method="log_standard"),
    TargetTransformSpec("Tempo_ate_P3", method="standard"),
]

pipe = PreprocessingPipeline(FATORES, alvos)

train_df, test_df = pipe.split(resultados, test_size=0.25, seed=SEED)   # 1º: particionar
pipe.fit(train_df)                                                      # 2º: ajustar só no treino

X_train = pipe.transform_X(train_df)
X_test = pipe.transform_X(test_df)
y_train = pipe.transform_y(train_df, "Tempo_total")
y_test = pipe.transform_y(test_df, "Tempo_total")

print(f"treino : {len(train_df):3d} linhas / {train_df['point_id'].nunique():3d} pontos")
print(f"teste  : {len(test_df):3d} linhas / {test_df['point_id'].nunique():3d} pontos")
print(f"X      : {X_train.shape} treino, {X_test.shape} teste")
print(f"colunas: {pipe.get_feature_names()}")

treino : 192 linhas /  96 pontos
teste  :  64 linhas /  32 pontos
X      : (192, 3) treino, (64, 3) teste
colunas: ['Tempo processo 1', 'Tempo processo 2', 'Tempo processo 3']


In [28]:
# a propriedade que torna a avaliação válida
vazamento = set(train_df["point_id"]) & set(test_df["point_id"])
print(f"pontos presentes nos DOIS lados: {len(vazamento)}   (tem de ser 0)")
print("réplicas por ponto no treino:", sorted(map(int, train_df.groupby('point_id').size().unique())))
print("réplicas por ponto no teste :", sorted(map(int, test_df.groupby('point_id').size().unique())))

# (a) entradas: ajustar em 1 linha == ajustar no treino inteiro
print("\ndeterminismo das entradas:",
      np.array_equal(pipe.transform_X(train_df.iloc[[0]]), X_train[[0]]))

# (b) réplicas do mesmo ponto produzem X idêntico
primeiro = train_df["point_id"].iloc[0]
X_rep = pipe.transform_X(train_df[train_df["point_id"] == primeiro])
print(f"as {len(X_rep)} réplicas do ponto {primeiro} dão X idêntico:", np.allclose(X_rep, X_rep[0]))

# (c) alvo: estatísticas vêm do TREINO, não do conjunto completo
alvo = pipe.targets_["Tempo_total"]
print(f"\nmédia guardada  : {alvo.mean_:.6f}")
print(f"média do treino : {np.log1p(train_df['Tempo_total'].to_numpy(float)).mean():.6f}  <- bate")
print(f"média de tudo   : {np.log1p(resultados['Tempo_total'].to_numpy(float)).mean():.6f}  <- NÃO bate (correto)")

# (d) a transformação do alvo é reversível
volta_y = pipe.inverse_transform_y(y_test, "Tempo_total")
print(f"\nerro máx. no round-trip: {np.abs(volta_y - test_df['Tempo_total'].to_numpy(float)).max():.2e}")

pontos presentes nos DOIS lados: 0   (tem de ser 0)
réplicas por ponto no treino: [2]
réplicas por ponto no teste : [2]

determinismo das entradas: True
as 2 réplicas do ponto 1 dão X idêntico: True

média guardada  : 11.627557
média do treino : 11.627557  <- bate
média de tudo   : 11.639851  <- NÃO bate (correto)

erro máx. no round-trip: 1.46e-10


Quanto custaria não agrupar? Basta medir.

In [29]:
from sklearn.model_selection import train_test_split, GridSearchCV, GroupKFold
from sed_meta.metamodels.tree_family import build_random_forest, GRID_RANDOM_FOREST
from sed_meta.metamodels.baseline import build_dummy_mean


def r2(y_true, y_pred):
    return 1 - np.sum((y_true - y_pred) ** 2) / np.sum((y_true - y_true.mean()) ** 2)


def rmse(y_true, y_pred):
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))


solto_tr, solto_te = train_test_split(resultados, test_size=0.25, random_state=SEED)
p_solto = PreprocessingPipeline(FATORES, alvos).fit(solto_tr)
m_solto = build_random_forest().fit(
    p_solto.transform_X(solto_tr), p_solto.transform_y(solto_tr, "Tempo_total")
)
r2_solto = r2(p_solto.transform_y(solto_te, "Tempo_total"),
              m_solto.predict(p_solto.transform_X(solto_te)))

print(f"pontos contaminados no teste: "
      f"{len(set(solto_tr['point_id']) & set(solto_te['point_id']))} "
      f"de {solto_te['point_id'].nunique()}")
print(f"R² com split SOLTO (inflado): {r2_solto:.4f}")

pontos contaminados no teste: 44 de 54
R² com split SOLTO (inflado): 0.8504


---
## 11. Metamodelo — Random Forest

Treina-se sobre as réplicas. Não se passa peso amostral: cada linha é uma observação de
peso um.

Duas referências acompanham o resultado, e nenhuma é dispensável — o preditor da média é o
**piso** (quem não o supera não aprendeu nada) e o teto de ruído é o máximo atingível.

> O modelo opera no espaço **transformado**. A avaliação formal é atribuição da Fase 4,
> ainda não implementada; aqui as métricas servem para confirmar que o treino funciona.

In [30]:
modelo = build_random_forest().fit(X_train, y_train)
piso = build_dummy_mean().fit(X_train, y_train)

resumo = pd.DataFrame([
    {"modelo": "Piso (média)",
     "R² treino": r2(y_train, piso.predict(X_train)),
     "R² teste": r2(y_test, piso.predict(X_test)),
     "RMSE teste": rmse(y_test, piso.predict(X_test))},
    {"modelo": "Random Forest",
     "R² treino": r2(y_train, modelo.predict(X_train)),
     "R² teste": r2(y_test, modelo.predict(X_test)),
     "RMSE teste": rmse(y_test, modelo.predict(X_test))},
]).set_index("modelo")

print(f"teto de ruído na escala em que o modelo treina (log): {teto_log['Tempo_total']:.4f}")
resumo.round(4)

teto de ruído na escala em que o modelo treina (log): 0.9205


,R² treino,R² teste,RMSE teste
modelo,,,
Piso (média),0.0000,-0.0340,0.8446
Random Forest,0.9441,0.7917,0.3791


### 11.1 O $R^2$ de treino passa do teto — e isso não é sobreajuste

Réplicas de um mesmo ponto têm $X$ idêntico, então nenhuma árvore consegue separá-las: a
floresta devolve **um** valor para aquele $X$, e a dispersão interna do ponto fica como
resíduo. O erro de treino é, portanto, limitado por baixo pelo próprio ruído do simulador.

Só que o limite exato desse argumento e o teto de ruído **não são o mesmo número**, e a
diferença é de graus de liberdade. Escrevendo $w = \mathrm{SQ}_{\text{intra}} /
\mathrm{SQ}_{\text{total}}$:

$$R^2_{\text{treino}} \;\le\; 1 - w
\qquad\text{enquanto}\qquad
\text{teto} \;=\; 1 - w\,\frac{N-1}{\sum_p (n_p - 1)}$$

Com $P$ pontos e $R$ réplicas cada, esse fator vale $\frac{RP-1}{P(R-1)} \approx
\frac{R}{R-1}$: cerca de $1{,}25$ com $R = 5$, mas **quase $2$ com $R = 2$**. Quanto menos
réplicas, mais conservador o teto fica em relação ao limite real — e mais fácil o $R^2$ de
treino aparecer acima dele sem que nada esteja errado. Com as duas réplicas desta rodada,
o teto chega a descontar praticamente o dobro do ruído que o limite real desconta.

O teto continua sendo a referência certa para o $R^2$ de **teste**, que é onde ele
importa. Mas ele é um **auxílio de diagnóstico, não uma garantia**: quem impede o erro de
avaliação é o split agrupado, não o teto.

In [31]:
# a conta, sobre as MESMAS linhas de treino e na MESMA escala do alvo
y_log = np.log1p(train_df["Tempo_total"].to_numpy(float))
g = train_df["point_id"].to_numpy()

sq_intra = sum(((y_log[g == p] - y_log[g == p].mean()) ** 2).sum() for p in np.unique(g))
sq_total = ((y_log - y_log.mean()) ** 2).sum()
w = sq_intra / sq_total
fator_dof = (len(y_log) - 1) / sum(int((g == p).sum()) - 1 for p in np.unique(g))

teto_treino = compute_noise_ceiling(
    train_df.assign(Tempo_total=y_log), ["Tempo_total"])["Tempo_total"]

pd.DataFrame({"valor": {
    "w = SQ_intra / SQ_total":            w,
    "fator de graus de liberdade":        fator_dof,
    "limite real do R² de treino (1-w)":  1 - w,
    "teto de ruído (1 - w x fator)":      1 - w * fator_dof,
    "teto via compute_noise_ceiling":     teto_treino,
    "R² de treino observado":             r2(y_train, modelo.predict(X_train)),
}}).round(4)

,valor
w = SQ_intra / SQ_total,0.0355
fator de graus de liberdade,1.9896
limite real do R² de treino (1-w),0.9645
teto de ruído (1 - w x fator),0.9293
teto via compute_noise_ceiling,0.9293
R² de treino observado,0.9441


### 11.2 Por que o split agrupado não era preciosismo

Vale confrontar os números lado a lado. O split solto produz um $R^2$ muito acima do
correto — ele não estava prevendo, estava recuperando linhas que já tinha visto.

Note que o teto **não flagra formalmente** o caso: o valor inflado fica abaixo dele, não
acima. Mais uma vez, o teto é diagnóstico; quem impede o erro é o split.

In [32]:
pd.DataFrame({"R² teste": {
    "split agrupado (correto)":  r2(y_test, modelo.predict(X_test)),
    "split solto (contaminado)": r2_solto,
    "teto de ruído":             teto_log["Tempo_total"],
}}).round(4)

,R² teste
split agrupado (correto),0.7917
split solto (contaminado),0.8504
teto de ruído,0.9205


### 11.3 Busca de hiperparâmetros

O grid sugerido para esta família **não inclui o número de árvores**: em *bagging* o erro
converge conforme se acrescentam árvores, sem sobreajustar, então buscar nesse eixo gasta
orçamento por uma resposta já conhecida.

A validação cruzada também precisa respeitar os grupos, pelo mesmo motivo da Seção 10 —
daí `GroupKFold` com os `point_id` do treino.

In [33]:
busca = GridSearchCV(build_random_forest(), GRID_RANDOM_FOREST,
                     cv=GroupKFold(n_splits=4), scoring="r2", n_jobs=-1)
busca.fit(X_train, y_train, groups=train_df["point_id"])

print("melhores hiperparâmetros:", busca.best_params_)
print(f"R² em validação cruzada (agrupada) : {busca.best_score_:.4f}")
print(f"R² no teste (hold-out)             : {r2(y_test, busca.best_estimator_.predict(X_test)):.4f}")

melhores hiperparâmetros: {'max_depth': None, 'max_features': 1.0, 'min_samples_leaf': 1}
R² em validação cruzada (agrupada) : 0.6480
R² no teste (hold-out)             : 0.7870


As previsões voltam à escala original — segundos — pela transformação inversa do alvo. É
disso que a Fase 4 precisará para reportar RMSE e MAE em unidades interpretáveis.

Cada ponto de teste aparece **duas vezes** no gráfico — uma por réplica — com o mesmo valor
previsto e valores observados diferentes. A dispersão vertical de cada par é o ruído do
simulador, que o metamodelo não pode nem deve reproduzir.

In [34]:
previsto = pipe.inverse_transform_y(busca.best_estimator_.predict(X_test), "Tempo_total") / 3600
observado = test_df["Tempo_total"].to_numpy(float) / 3600
lim = [min(observado.min(), previsto.min()) * 0.9, max(observado.max(), previsto.max()) * 1.1]

par = go.Figure()
par.add_trace(go.Scatter(x=lim, y=lim, mode="lines", name="previsão perfeita",
                         line=dict(color="gray", dash="dash", width=1)))
par.add_trace(go.Scatter(
    x=observado, y=previsto, mode="markers", name="execuções de teste",
    marker=dict(size=7, color=CORES[ESCOLHIDO], opacity=0.7,
                line=dict(width=0.5, color="white")),
    hovertemplate="observado: %{x:.2f} h<br>previsto: %{y:.2f} h<extra></extra>",
))
par.update_layout(
    title=dict(text="Random Forest — escala original", x=0.5),
    xaxis=dict(title="Tempo_total observado (h)", type="log"),
    yaxis=dict(title="Tempo_total previsto (h)", type="log"),
    width=560, height=520, margin=dict(l=60, r=20, t=50, b=50),
    plot_bgcolor="rgba(245,245,247,0.6)",
)
par.show()

print(f"MAE na escala original: {np.mean(np.abs(previsto - observado)):.2f} h")

MAE na escala original: 3.36 h


---
## Resumo

| Etapa | Verificado |
|---|---|
| Cockpit exemplar | Formato auditado: 1 id + 10 entradas + 13 saídas; entradas todas inteiras, 6 constantes, 1 índice disfarçado de fator, `Replication ID` vazia |
| `Replication ID` | Semântica fixada — id do ponto de origem, constante entre réplicas; equivale ao `point_id` interno |
| Contrato de entrada | Derivado do arquivo (não redigitado) e verificado por `assert` contra os fatores declarados |
| Espaço de parâmetros | 3 fatores inteiros com faixas declaradas, 6 entradas fixas explicitadas |
| Amostragem | Monte Carlo, LHS e Sobol com $n = 128$ (potência de 2); os três genuinamente distintos |
| Preenchimento do espaço | 7 métricas; a marginal satura em espaço inteiro, a discriminação vai para as conjuntas |
| Inspeção visual | Três nuvens 3D interativas sobre o cubo normalizado, mais a sobreposição |
| Escolha do método | Por posto médio nas 7 métricas — critério explícito, não preferência |
| Replicações | 128 × 2 = 256 execuções, sementes determinísticas, `dtypes` preservados |
| Cockpit gravado | `runs/test_run_scenario_01_120926/`, bloco de saída vazio, round-trip conferido (256 `pendente`) |
| Simulação (fingida) | Resposta de linha serial com gargalo, ruído lognormal heterocedástico |
| Teto de ruído | Calculado na escala do treino, com a correção de graus de liberdade explicitada |
| Pré-processamento | Split **agrupado por ponto**, entradas determinísticas, alvo sem vazamento |
| Metamodelo | Random Forest sobre réplicas, `GroupKFold` na busca, volta à escala original |

**Fora deste notebook, por não estarem implementadas:** a avaliação estatística formal
(Fase 4) e a camada de explicabilidade (Fase 5).